In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
print("Shape:", df.shape)
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Target: Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df.drop(columns=['Order_ID']).copy()

In [ ]:
# Task 2: Write your code here:
# Missing values
print("Missing values:")
print(df_clean.isnull().sum())



In [ ]:
# numeric
df_clean = df_clean.dropna(subset=['Courier_Experience_yrs', 'Delivery_Time'])
print(f"After dropping missing values: {df_clean.shape}")

# categorical
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

In [ ]:
# Task 3: Write your code here:

#Check and remove duplicates if any exist
print(f"Before dropping duplicates: {df_clean.shape}")
df_clean = df_clean.drop_duplicates()
print(f"After dropping duplicates: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print("Shape after one-hot encoding:", df_clean.shape)
display(df_clean.head())


In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)
scaler = StandardScaler()
df_clean = pd.DataFrame(scaler.fit_transform(df_clean), columns=df_clean.columns)
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

In [ ]:
# Task 1: Write your code here:
# Split the dataset into features (X) and target (y)
feature_cols = [col for col in df_clean.columns if col != 'Delivery_Time']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
# Use the correct split: KFold OR StratifiedKFold
# Train a RandomForest model
# Evaluate using MAE (Mean Absolute Error) ONLY
# Print the averaged score across all folds

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train_fold, y_train_fold)
    predictions = model.predict(X_test_fold)
    mae = mean_absolute_error(y_test_fold, predictions)
    mae_scores.append(mae)
    print(f"Fold {fold+1} MAE: {mae:.2f}")

print(f"\nAverage MAE across all folds: {np.mean(mae_scores):.2f}")


In [ ]:
final_model = RandomForestRegressor(random_state=42)
final_model.fit(X, y)

feature_importances = final_model.feature_importances_
features = X.columns

importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance from RandomForestRegressor')
plt.gca().invert_yaxis() # To have the most important feature at the top
plt.show()

In [ ]:
# Task 2: Write your code here:

predicted_delivery_times = final_model.predict(X)

plt.figure(figsize=(10, 5))
plt.hist(predicted_delivery_times, bins=30, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Predicted Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:


In [ ]:
from catboost import CatBoostRegressor
print("CatBoostRegressor imported successfully.")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train_fold, y_train_fold)
    rf_predictions = rf_model.predict(X_test_fold)

    cat_model = CatBoostRegressor(verbose=0, random_state=42)
    cat_model.fit(X_train_fold, y_train_fold)
    cat_predictions = cat_model.predict(X_test_fold)

    averaged_predictions = (rf_predictions + cat_predictions) / 2

    mae = mean_absolute_error(y_test_fold, averaged_predictions)
    ensemble_mae_scores.append(mae)
    print(f"Fold {fold+1} Ensemble MAE: {mae:.2f}")


In [ ]:
print(f"\nAverage Ensemble MAE across all folds: {np.mean(ensemble_mae_scores):.2f}")